# CRAFT-GC — Image-Level Fairness Pipeline

**Runtime:** T4 GPU required. **Secrets:** `HF_TOKEN` for HuggingFace models.

| Cell | Task | Time |
|------|------|------|
| 1 | Setup + clone | ~5 min |
| 2 | Stage A text embeddings (optional) | ~2 min |
| 3 | GPU images + CLIP eval + figures | **2–4 hours** (50 prompts) |
| 4 | Download results | ~1 min |

In [ ]:
# CELL 1 — Setup
import os, sys, shutil, subprocess
from getpass import getpass

os.chdir('/content')
REPO = 'https://github.com/Natiabay/Dynamic-Open-Set-Post-Processing-for-Intersectional-Fairness-in-Text-to-Image-Diffusion-Models.git'
PROJECT = '/content/Research'

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'torch', 'torchvision', 'open-clip-torch', 'diffusers', 'transformers',
    'accelerate', 'pandas', 'matplotlib', 'seaborn', 'tqdm', 'huggingface_hub'], check=True)

from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
except Exception:
    login(token=getpass('HF token: '))

if os.path.isdir(PROJECT):
    shutil.rmtree(PROJECT)
subprocess.run(['git', 'clone', REPO, PROJECT], cwd='/content', check=True)

os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', PROJECT], check=True)

import torch, craft_gc
assert torch.cuda.is_available(), 'Enable T4 GPU runtime'
print('SETUP OK | GPU:', torch.cuda.get_device_name(0), '| craft_gc:', craft_gc.__version__)

In [ ]:
# CELL 2 — Stage A (CLIP text embeddings, optional)
!python scripts/evaluate_embeddings.py --device cuda

In [ ]:
# CELL 3 — GPU image generation + evaluation + figures
import subprocess, sys

result = subprocess.run([
    sys.executable, "scripts/run_pilot_images.py",
    "--device", "cuda",
    "--limit", "50",
    "--seeds", "42", "123", "456",
    "--methods", "base", "prompt_aug", "fairimagen", "craftgc",
    "--model_id", "SG161222/Realistic_Vision_V5.1_noVAE",
    "--vae_id", "stabilityai/sd-vae-ft-mse"
], capture_output=False, text=True)

result2 = subprocess.run([
    sys.executable, "scripts/evaluate_images.py",
    "--device", "cuda",
    "--manifest", "results/pilot_images/manifest.json",
    "--output", "results/image_eval_v2.csv"
], capture_output=False, text=True)

subprocess.run([sys.executable, "scripts/merge_paper_results.py"])
subprocess.run([sys.executable, "scripts/plot_figures.py"])
subprocess.run([sys.executable, "scripts/make_qualitative_grid.py"])

print("\nAll done. Download results/ and craft-gc-paper/figures/")

In [ ]:
# CELL 4 — Download
import shutil
from google.colab import files
shutil.make_archive('/content/craft_gc_results', 'zip', 'results')
shutil.make_archive('/content/craft_gc_figures', 'zip', 'craft-gc-paper/figures')
files.download('/content/craft_gc_results.zip')
files.download('/content/craft_gc_figures.zip')
print('Done. Unzip into ~/Desktop/Research/ on your PC.')